In [1]:
"""
Schritt 1-3 mit ECHTEN Daten: Ist-Daten (opentransportdata.swiss) + Wetter (Open-Meteo) -> SQLite

Vorbereitung (einmalig, von Hand):
  1. Ladet Tagesdateien (CSV, oder die Monats-ZIPs) vom Archiv herunter:
     https://archive.opentransportdata.swiss/
     und legt sie in den Ordner DATA_DIR (siehe unten). ZIP-Dateien muesst ihr nicht entpacken.
  2. python build_dataset.py

Das Skript erfindet NIEMALS Daten: Fehlt etwas, bricht es mit einer Fehlermeldung ab.
"""
import glob
import io
import os
import sqlite3
import zipfile

import pandas as pd
import requests

# ------------------------------------------------------------------ Einstellungen
DATA_DIR = "ist_daten"                    # Ordner mit den heruntergeladenen CSV- oder ZIP-Dateien
DB_NAME = "project_transit_weather.db"
TABLE = "transit_weather_data"

CITIES = {  # Stadt: (lat, lon, Hauptbahnhof-Haltestelle)
    "Zürich":     (47.3769, 8.5417, "Zürich HB"),
    "Bern":       (46.9480, 7.4474, "Bern"),
    "Basel":      (47.5596, 7.5886, "Basel SBB"),
    "Genf":       (46.2044, 6.1432, "Genève"),
    "Lausanne":   (46.5197, 6.6323, "Lausanne"),
    "Winterthur": (47.5001, 8.7244, "Winterthur"),
}
# Stadt-Zuordnung: Bahnhof-Name ODER Bus/Tram-Haltestellen nach Schweizer Muster "Ort, Haltestelle"
# (Genf heisst in den Daten "Genève", Basel hat auch "Basel, ..." Haltestellen)
CITY_PREFIX = {"Zürich": "Zürich, ", "Bern": "Bern, ", "Basel": "Basel, ",
               "Genf": "Genève, ", "Lausanne": "Lausanne, ", "Winterthur": "Winterthur, "}

ONLY_DAYS = None   # None = alle Dateien. Schneller: nur bestimmte Tage lesen, z.B. die ersten 7 Tage jedes Monats 2024:
# ONLY_DAYS = [f"2024-{m:02d}-{d:02d}" for m in range(1, 13) for d in range(1, 8)]   # 84 Tage, alle Jahreszeiten
MODES = {
    "Zug": "Train", "Train": "Train", "S": "Train",
    "Tram": "Tram", "Strassenbahn": "Tram", "T": "Tram",
    "Metro": "Tram",  # Lausanne Metro als Tram/Schienenverkehr werten
    "Bus": "Bus", "B": "Bus"
}  # PRODUKT_ID -> mode
HOUR_FROM, HOUR_TO = 6, 21               # nur Tagesbetrieb
MAX_ROWS_PER_CITY_DAY = 100              # Stichprobe: zufaellig, max. N Halte pro Stadt, VERKEHRSMITTEL und Tag (so bleiben Bus/Tram sichtbar)
RANDOM_STATE = 42
DELAY_LIMIT_MIN = 3                      # verspaetet = 3 Minuten oder mehr

USECOLS = ["BETRIEBSTAG", "FAHRT_BEZEICHNER", "PRODUKT_ID", "LINIEN_TEXT", "FAELLT_AUS_TF",
           "HALTESTELLEN_NAME", "ANKUNFTSZEIT", "AN_PROGNOSE", "AN_PROGNOSE_STATUS",
           "ABFAHRTSZEIT", "AB_PROGNOSE", "AB_PROGNOSE_STATUS", "DURCHFAHRT_TF"]


# ------------------------------------------------------------------ Hilfsfunktionen
def _wanted(name):
    return ONLY_DAYS is None or any(d in name for d in ONLY_DAYS)


def _open_sources():
    """Liefert (Name, open_fn) fuer jede CSV (auch in ZIP-Dateien). open_fn() gibt einen frischen Stream zurueck."""
    paths = sorted(glob.glob(os.path.join(DATA_DIR, "*")))
    if not paths:
        raise SystemExit(f"Keine Dateien in '{DATA_DIR}'. Bitte Ist-Daten dort ablegen.")
    for p in paths:
        if p.lower().endswith(".zip"):
            with zipfile.ZipFile(p) as z:
                for n in sorted(z.namelist()):
                    if n.lower().endswith(".csv") and _wanted(n):
                        yield f"{os.path.basename(p)}:{n}", (lambda p=p, n=n: zipfile.ZipFile(p).open(n))
        elif p.lower().endswith(".csv") and _wanted(p):
            yield os.path.basename(p), (lambda p=p: open(p, "rb"))


def _sep(stream):
    head = stream.readline().decode("utf-8", errors="replace")
    stream.close()
    return ";" if head.count(";") >= head.count("\t") else "\t"


def _city_of(stop_names):
    """Ordnet Haltestellennamen einer Stadt zu (oder NaN)."""
    s = stop_names.astype("string")
    out = pd.Series(pd.NA, index=s.index, dtype="string")
    for city, (_, _, hub) in CITIES.items():
        m = (s == hub) | s.str.startswith(CITY_PREFIX[city], na=False)
        out = out.mask(m & out.isna(), city)
    return out


def _parse(col):
    return pd.to_datetime(col, format="mixed", dayfirst=True, errors="coerce")


def _chunks(open_fn, sep):
    """Liest die Datei in Teilen (Stream). UTF-8, bei Zeichenfehler im ersten Teil Latin-1."""
    for enc in ("utf-8", "latin-1"):
        reader = pd.read_csv(open_fn(), sep=sep, usecols=USECOLS, dtype=str, encoding=enc, chunksize=250_000)
        try:
            first = next(reader)
        except UnicodeDecodeError:
            continue
        yield first
        yield from reader
        return


def load_transit():
    stats = {"gelesen": 0, "in_unseren_staedten": 0, "gueltige_modes": 0, "nicht_ausgefallen": 0,
             "status_REAL": 0, "mit_zeiten": 0, "im_zeitfenster": 0}
    parts, produkte = [], pd.Series(dtype="int64")
    for name, open_fn in _open_sources():
        sep = _sep(open_fn())
        print(f"Lese {name} ...", flush=True)
        n_file = 0
        file_parts = []
        for df in _chunks(open_fn, sep):
            stats["gelesen"] += len(df)
            df["city"] = _city_of(df["HALTESTELLEN_NAME"])
            df = df[df["city"].notna()]
            stats["in_unseren_staedten"] += len(df)
            produkte = produkte.add(df["PRODUKT_ID"].value_counts(), fill_value=0)
            df = df[df["PRODUKT_ID"].isin(MODES)]
            stats["gueltige_modes"] += len(df)
            df = df[(df["FAELLT_AUS_TF"].str.lower() != "true") & (df["DURCHFAHRT_TF"].str.lower() != "true")]
            stats["nicht_ausgefallen"] += len(df)

            # Ankunft bevorzugen, sonst Abfahrt (Endhalt hat keine Abfahrt, Starthalt keine Ankunft)
            use_arr = df["AN_PROGNOSE_STATUS"].eq("REAL") & df["ANKUNFTSZEIT"].notna()
            use_dep = ~use_arr & df["AB_PROGNOSE_STATUS"].eq("REAL") & df["ABFAHRTSZEIT"].notna()
            df = df[use_arr | use_dep].copy()
            use_arr = use_arr.loc[df.index]
            stats["status_REAL"] += len(df)
            plan = _parse(df["ANKUNFTSZEIT"].where(use_arr, df["ABFAHRTSZEIT"]))
            actual = _parse(df["AN_PROGNOSE"].where(use_arr, df["AB_PROGNOSE"]))
            df["planned"] = plan
            df["delay_raw"] = (actual - plan).dt.total_seconds() / 60
            df = df.dropna(subset=["planned", "delay_raw"])
            stats["mit_zeiten"] += len(df)
            df["mode"] = df["PRODUKT_ID"].map(MODES)
            df["line"] = df["LINIEN_TEXT"]
            file_parts.append(df[["city", "planned", "line", "mode", "delay_raw"]])
            n_file += len(df)
        # Stichprobe SOFORT pro Datei (= pro Tag) ziehen, damit der Speicher auch bei einem ganzen Jahr reicht
        if file_parts:
            f = pd.concat(file_parts, ignore_index=True)
            f = f[(f["planned"].dt.hour >= HOUR_FROM) & (f["planned"].dt.hour <= HOUR_TO)]
            stats["im_zeitfenster"] += len(f)
            f = f.sample(frac=1, random_state=RANDOM_STATE)
            parts.append(f[f.groupby(["city", "mode"]).cumcount() < MAX_ROWS_PER_CITY_DAY])
        print(f"  {name}: {n_file} verwertbare Halte, {len(parts[-1]) if file_parts else 0} in der Stichprobe", flush=True)

    if not parts:
        raise SystemExit("Keine verwertbaren Zeilen gefunden (Status REAL, eure Staedte). "
                         "Pruefen: Haltestellennamen (CITIES) und Status-Spalten.")
    print("\nPRODUKT_ID in euren Staedten (vor dem Filter):\n", produkte.astype(int).sort_values(ascending=False).to_string())
    t = pd.concat(parts, ignore_index=True)
    t["date"] = t["planned"].dt.date
    t["hour"] = t["planned"].dt.hour
    t = t[t.groupby(["city", "date", "mode"]).cumcount() < MAX_ROWS_PER_CITY_DAY]   # Obergrenze je Stadt und Tag (Datum nach Soll-Zeit)
    stats["nach_stichprobe"] = len(t)
    t["delay_min"] = t["delay_raw"].clip(lower=0).round(2)   # zu frueh = 0 Minuten Verspaetung
    t["delayed"] = (t["delay_min"] >= DELAY_LIMIT_MIN).astype(int)
    return t.drop(columns=["planned"]), stats


import time
import requests
import pandas as pd

def fetch_weather(dates):
    start_min, end_max = min(dates), max(dates)
    frames = []
    
    # Alle Monate ermitteln, die in unserer Stichprobe vorkommen
    months = sorted(list(set(d.replace(day=1) for d in dates)))
    
    for city, (lat, lon, _) in CITIES.items():
        print(f"Hole Wetterdaten für {city} (monatsweise)...", flush=True)
        city_frames = []
        
        for m_start in months:
            m_start_date = m_start.date() if hasattr(m_start, 'date') else m_start
            m_end = (pd.to_datetime(m_start_date) + pd.offsets.MonthEnd(0)).date()
            
            s_date = max(m_start_date, start_min)
            e_date = min(m_end, end_max)
            if s_date > e_date:
                continue
                
            url = ("https://archive-api.open-meteo.com/v1/archive?"
                   f"latitude={lat}&longitude={lon}&start_date={s_date}&end_date={e_date}"
                   "&hourly=temperature_2m,precipitation,snowfall&timezone=Europe%2FZurich")
            
            # 5 Versuche mit jeweils 10 Sekunden Pause bei Server-Überlastung (503)
            for attempt in range(5):
                try:
                    r = requests.get(url, timeout=45)
                    r.raise_for_status()
                    break
                except Exception as e:
                    if attempt == 4:
                        print(f"Fehler bei {city} für {s_date} bis {e_date}: {e}")
                        raise e
                    print(f"Server-Aussetzer (503) bei {city}. Warte 10 Sek. vor Versuch {attempt+2}/5...")
                    time.sleep(10)
            
            h = r.json().get("hourly", {})
            if "time" in h:
                w = pd.DataFrame({
                    "timestamp": pd.to_datetime(h["time"]), 
                    "temperature": h["temperature_2m"],
                    "precipitation": h["precipitation"], 
                    "snowfall": h["snowfall"]
                })
                w["date"] = w["timestamp"].dt.date
                w["hour"] = w["timestamp"].dt.hour
                w["city"] = city
                city_frames.append(w)
            
            time.sleep(2) # Etwas längere Pause zwischen den Anfragen, um den Server zu schonen
            
        if city_frames:
            frames.append(pd.concat(city_frames, ignore_index=True))
            
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def main(weather_fn=fetch_weather):
    print("--- SCHRITT 1: ÖV-ISTDATEN EINLESEN ---")
    transit, stats = load_transit()
    print("\n--- SCHRITT 2: WETTER + JOIN ---")
    weather = weather_fn(set(transit["date"]))
    merged = transit.merge(weather, on=["city", "date", "hour"], how="left")
    n_before = len(merged)
    merged = merged.dropna(subset=["temperature", "precipitation", "snowfall"])
    stats["join_ohne_wetter_verloren"] = n_before - len(merged)
    stats["final"] = len(merged)
    print("\nDokumentation der Datenmenge (fuer die Folien):")
    for k, v in stats.items():
        print(f"  {k:30s} {v:>10,}")

    print("\n--- SCHRITT 3: SQLITE ---")
    with sqlite3.connect(DB_NAME) as conn:
        merged.drop(columns=["delay_raw"]).to_sql(TABLE, conn, if_exists="replace", index=False)
        print(pd.read_sql(f"SELECT city, mode, COUNT(*) AS n, ROUND(AVG(delayed),3) AS anteil_verspaetet, "
                          f"ROUND(AVG(precipitation),2) AS regen FROM {TABLE} GROUP BY city, mode", conn))
    return merged


if __name__ == "__main__":
    main()

--- SCHRITT 1: ÖV-ISTDATEN EINLESEN ---
Lese 2024-01-01_istdaten.csv ...
  2024-01-01_istdaten.csv: 4533 verwertbare Halte, 700 in der Stichprobe
Lese 2024-01-02_istdaten.csv ...
  2024-01-02_istdaten.csv: 4490 verwertbare Halte, 700 in der Stichprobe
Lese 2024-01-03_istdaten.csv ...
  2024-01-03_istdaten.csv: 4678 verwertbare Halte, 700 in der Stichprobe
Lese 2024-01-04_istdaten.csv ...
  2024-01-04_istdaten.csv: 4668 verwertbare Halte, 700 in der Stichprobe
Lese 2024-01-05_istdaten.csv ...
  2024-01-05_istdaten.csv: 4733 verwertbare Halte, 700 in der Stichprobe
Lese 2024-01-06_istdaten.csv ...
  2024-01-06_istdaten.csv: 4618 verwertbare Halte, 700 in der Stichprobe
Lese 2024-01-07_istdaten.csv ...
  2024-01-07_istdaten.csv: 4522 verwertbare Halte, 700 in der Stichprobe
Lese 2024-01-08_istdaten.csv ...
  2024-01-08_istdaten.csv: 4723 verwertbare Halte, 700 in der Stichprobe
Lese 2024-01-09_istdaten.csv ...
  2024-01-09_istdaten.csv: 4666 verwertbare Halte, 700 in der Stichprobe
Lese 2

In [2]:
import glob
import pandas as pd

# Prüfe die ersten CSV-Dateien im Ordner
for path in glob.glob("ist_daten/**/*.csv", recursive=True)[:3]:
    print(f"Lese Vorschau von: {path}")
    df = pd.read_csv(path, sep=None, engine="python", nrows=50000, dtype=str)
    if "PRODUKT_ID" in df.columns:
        print(df["PRODUKT_ID"].value_counts(dropna=False))
        break

Lese Vorschau von: ist_daten/2024-02-14_istdaten.csv
PRODUKT_ID
Zug    49573
Bus      399
NaN       28
Name: count, dtype: int64
